# House Price - Experiment v2

PyTorch MLP regression using the Ames Housing dataset. This notebook keeps the v1 preprocessing contract while replacing XGBoost with a reproducible PyTorch training and evaluation pipeline.

In [ ]:
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'house_price' / 'data' / 'train.csv').exists():
            return candidate
        if (candidate / 'data' / 'train.csv').exists() and candidate.name == 'house_price':
            return candidate.parent
    raise FileNotFoundError('Could not locate house_price/data/train.csv from the current notebook context.')

PROJECT_ROOT = find_project_root()
HOUSE_PRICE_ROOT = PROJECT_ROOT / 'house_price'
DATA_DIR = HOUSE_PRICE_ROOT / 'data'
EXPERIMENT_DIR = HOUSE_PRICE_ROOT / 'experiments' / 'house_price_v2'
IMAGES_DIR = EXPERIMENT_DIR / 'images'
SUBMISSION_DIR = HOUSE_PRICE_ROOT / 'submissions'
BEST_MODEL_PATH = EXPERIMENT_DIR / 'best_model.pt'
PARAMS_PATH = EXPERIMENT_DIR / 'params.json'
TUNING_RESULTS_PATH = EXPERIMENT_DIR / 'tuning_results.csv'
SUBMISSION_PATH = SUBMISSION_DIR / 'house_price_v2_submission.csv'
for directory in (EXPERIMENT_DIR, IMAGES_DIR, SUBMISSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

params = {
    'data': {
        'input_dir': str(DATA_DIR),
        'train_file': 'train.csv',
        'test_file': 'test.csv',
        'target': 'SalePrice',
        'validation_size': 0.2,
        'random_state': 42
    },
    'experiment': {
        'name': 'house_price_v2',
        'best_model_file': str(BEST_MODEL_PATH),
        'params_file': str(PARAMS_PATH),
        'tuning_results_file': str(TUNING_RESULTS_PATH),
        'images_dir': str(IMAGES_DIR),
        'submission_file': str(SUBMISSION_PATH),
        'figure_dpi': 150
    },
    'train': {
        'hidden1': 128,
        'hidden2': 64,
        'learning_rate': 0.001,
        'batch_size': 32,
        'epochs': 100
    },
    'tuning': [
        {'name': 'small', 'hidden1': 64, 'hidden2': 32, 'learning_rate': 0.001, 'batch_size': 32, 'epochs': 100},
        {'name': 'medium', 'hidden1': 128, 'hidden2': 64, 'learning_rate': 0.001, 'batch_size': 32, 'epochs': 100},
        {'name': 'large', 'hidden1': 256, 'hidden2': 128, 'learning_rate': 0.001, 'batch_size': 32, 'epochs': 100}
    ],
    'evaluation': {'metric': 'RMSE'}
}
with PARAMS_PATH.open('w', encoding='utf-8') as file:
    json.dump(params, file, indent=4)

print(f'Project root: {PROJECT_ROOT}')
print(f'Experiment directory: {EXPERIMENT_DIR}')

## Load and preprocess data

The missing-value handling, dropped columns, combined one-hot encoding, and `SalePrice` target are retained from v1.

In [ ]:
train_df = pd.read_csv(DATA_DIR / params['data']['train_file'])
test_df = pd.read_csv(DATA_DIR / params['data']['test_file'])
train_ids = train_df['Id'].copy()

train_categorical = ['FireplaceQu', 'GarageType', 'GarageFinish', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageQual', 'GarageCond']
train_numeric = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']
test_categorical = ['FireplaceQu', 'GarageType', 'GarageFinish', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageQual', 'GarageCond', 'MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 'KitchenQual', 'Functional', 'SaleType']
test_numeric = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']

for column in train_categorical:
    train_df[column] = train_df[column].fillna(train_df[column].mode()[0])
for column in train_numeric:
    train_df[column] = train_df[column].fillna(train_df[column].mean())
for column in test_categorical:
    test_df[column] = test_df[column].fillna(test_df[column].mode()[0])
for column in test_numeric:
    test_df[column] = test_df[column].fillna(test_df[column].mean())

for column in ['Id', 'Alley', 'PoolQC', 'Fence', 'MiscFeature']:
    train_df.drop(column, axis=1, inplace=True)
    test_df.drop(column, axis=1, inplace=True)

combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)
categorical_columns = ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'SaleType', 'SaleCondition']
combined = pd.get_dummies(combined, columns=categorical_columns, drop_first=True)
combined = combined.loc[:, ~combined.columns.duplicated()]

processed_train = combined.iloc[:len(train_df)].copy()
processed_test = combined.iloc[len(train_df):].drop(columns=[params['data']['target']]).copy()
features = processed_train.drop(columns=[params['data']['target']]).astype(np.float32)
target = processed_train[params['data']['target']].astype(np.float32)
features = features.fillna(features.mean()).fillna(0.0)
processed_test = processed_test.reindex(columns=features.columns, fill_value=0).astype(np.float32)
processed_test = processed_test.fillna(features.mean()).fillna(0.0)

print(f'Train features: {features.shape}')
print(f'Test features: {processed_test.shape}')

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(params['data']['random_state'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x_train, x_valid, y_train, y_valid = train_test_split(
    features.to_numpy(), target.to_numpy(),
    test_size=params['data']['validation_size'],
    random_state=params['data']['random_state']
)

train_dataset = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train).unsqueeze(1))
valid_tensor = torch.from_numpy(x_valid).to(device)
valid_target = torch.from_numpy(y_valid).unsqueeze(1).to(device)
print(f'Device: {device}')

## PyTorch model and training

In [ ]:
class HousePriceMLP(nn.Module):
    def __init__(self, input_size, hidden1, hidden2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 1)
        )

    def forward(self, inputs):
        return self.network(inputs)

def rmse(predictions, targets):
    return torch.sqrt(torch.mean((predictions - targets) ** 2)).item()

def train_model(config, seed):
    set_seed(seed)
    model = HousePriceMLP(features.shape[1], config['hidden1'], config['hidden2']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    criterion = nn.MSELoss()
    loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    history = []
    best_rmse = float('inf')
    best_state = None
    for epoch in range(config['epochs']):
        model.train()
        for batch_features, batch_target in loader:
            batch_features = batch_features.to(device)
            batch_target = batch_target.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_features), batch_target)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            validation_rmse = rmse(model(valid_tensor), valid_target)
        history.append({'epoch': epoch + 1, 'validation_rmse': validation_rmse})
        if validation_rmse < best_rmse:
            best_rmse = validation_rmse
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history, best_rmse

In [ ]:
tuning_rows = []
best_result = None
for config in params['tuning']:
    started = time.perf_counter()
    model, history, validation_rmse = train_model(config, params['data']['random_state'])
    result = {**config, 'validation_rmse': validation_rmse, 'training_seconds': time.perf_counter() - started}
    tuning_rows.append(result)
    if best_result is None or validation_rmse < best_result['validation_rmse']:
        best_result = result
        best_model = model
        best_history = history

tuning_results = pd.DataFrame(tuning_rows).sort_values('validation_rmse')
tuning_results.to_csv(TUNING_RESULTS_PATH, index=False)

checkpoint = {
    'model_state_dict': best_model.state_dict(),
    'input_size': features.shape[1],
    'model_config': {key: best_result[key] for key in ('hidden1', 'hidden2')},
    'preprocessing_columns': list(features.columns),
    'best_validation_rmse': best_result['validation_rmse']
}
torch.save(checkpoint, BEST_MODEL_PATH)
print(tuning_results)
print(f'Saved checkpoint: {BEST_MODEL_PATH}')

In [ ]:
history_df = pd.DataFrame(best_history)
figure = plt.figure(figsize=(8, 4))
plt.plot(history_df['epoch'], history_df['validation_rmse'])
plt.xlabel('Epoch')
plt.ylabel('Validation RMSE')
plt.title('House Price v2 validation curve')
plt.tight_layout()
figure.savefig(IMAGES_DIR / 'validation_rmse.png', dpi=params['experiment']['figure_dpi'])
plt.show()

loaded_checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
loaded_model = HousePriceMLP(loaded_checkpoint['input_size'], **loaded_checkpoint['model_config']).to(device)
loaded_model.load_state_dict(loaded_checkpoint['model_state_dict'])
loaded_model.eval()
with torch.no_grad():
    test_predictions = loaded_model(torch.from_numpy(processed_test.to_numpy()).to(device)).squeeze(1).cpu().numpy()

test_original = pd.read_csv(DATA_DIR / params['data']['test_file'])
submission = pd.DataFrame({'Id': test_original['Id'], 'SalePrice': test_predictions})
submission.to_csv(SUBMISSION_PATH, index=False)
best_validation_rmse = loaded_checkpoint['best_validation_rmse']
print(f'Best validation RMSE: {best_validation_rmse:.4f}')
print(f'Saved submission: {SUBMISSION_PATH}')
print(submission.head())